In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/akshat9303/embeddings/svd.pt
/kaggle/input/datasets/akshat9303/embeddings/skipgram.pt
/kaggle/input/datasets/akshat9303/embeddings/glove.pt


In [2]:
import nltk
from nltk.corpus import brown
import random
from collections import Counter
import os
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


nltk.download('brown')
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tagged_sents = list(brown.tagged_sents(tagset='universal'))
print(f"Total tagged sentences: {len(tagged_sents)}")

random.shuffle(tagged_sents)

num_sentences = len(tagged_sents)
train_sents = tagged_sents[:int(0.8 * num_sentences)]
validation_sents = tagged_sents[int(0.8 * num_sentences):int(0.9 * num_sentences)]
test_sents = tagged_sents[int(0.9 * num_sentences):]

print(f"Training sentences: {len(train_sents)}")
print(f"Validation sentences: {len(validation_sents)}")
print(f"Test sentences: {len(test_sents)}")

all_words = []
all_tags = []
for sent in train_sents:
    for word, tag in sent:
        all_words.append(word.lower())
        all_tags.append(tag)

word_counts = Counter(all_words)
vocab = {word for word, count in word_counts.items() if count >= 2}
vocab = sorted(vocab)

vocab = ['<PAD>'] + vocab

word_to_index = {word: idx for idx, word in enumerate(vocab)}
index_to_word = {idx: word for word, idx in word_to_index.items()}

unique_tags = sorted(set(all_tags))
tag_to_index = {tag: idx for idx, tag in enumerate(unique_tags)}
index_to_tag = {idx: tag for tag, idx in tag_to_index.items()}

print(f"Vocabulary size: {len(vocab)}")
print(f"Tag set size: {len(unique_tags)}")
print(f"Tags: {unique_tags}")

window_size = 2
print(f"Context window size: {window_size}")

def create_context_window(sentences, word_to_index, tag_to_index, window_size):
    X = []
    y = []

    pad_idx = word_to_index['<PAD>']
    for sentence in sentences:
        words = [word.lower() for word, tag in sentence]
        tags = [tag for word, tag in sentence]

        indexed_words = [word_to_index.get(word, pad_idx) for word in words]

        for i in range(len(words)):
            context = []
            for j in range(i - window_size, i + window_size + 1):
                if j < 0 or j >= len(words):
                    context.append(pad_idx)
                else:
                    context.append(indexed_words[j])
            
            X.append(context)
            y.append(tag_to_index[tags[i]])
    
    return X, y

X_train, y_train = create_context_window(train_sents, word_to_index, tag_to_index, window_size)
X_val, y_val = create_context_window(validation_sents, word_to_index, tag_to_index, window_size)
X_test, y_test = create_context_window(test_sents, word_to_index, tag_to_index, window_size)

print("Train examples:", len(X_train))
print("Validation examples:", len(X_val))
print("Test examples:", len(X_test))

X_train = torch.LongTensor(X_train)
y_train = torch.LongTensor(y_train)

X_val = torch.LongTensor(X_val)
y_val = torch.LongTensor(y_val)

X_test = torch.LongTensor(X_test)
y_test = torch.LongTensor(y_test)

print(X_train.shape)
print(y_train.shape)
print("Data preparation complete.")

class POSDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
train_dataset = POSDataset(X_train, y_train)
val_dataset = POSDataset(X_val, y_val)
test_dataset = POSDataset(X_test, y_test)

batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


class MLPPosTagger(nn.Module):
    def __init__(self, embedding_matrix, num_tags, window_size, hidden_dim=256, freeze_embeddings=True):
        super().__init__()

        vocab_size, embedding_dim = embedding_matrix.shape
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(embedding_matrix)

        if freeze_embeddings:
            self.embedding.weight.requires_grad_ = False
        
        self.window_size = window_size
        self.embedding_dim = embedding_dim
        self.input_dim = (2 * window_size + 1) * embedding_dim

        self.fc1 = nn.Linear(self.input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_tags)

    def forward(self, x):
        embedding = self.embedding(x)
        embedding = embedding.view(embedding.size(0), -1)

        out = self.fc1(embedding)
        out = self.relu(out)
        out = self.fc2(out)
        return out
    

[nltk_data] Downloading package brown to /usr/share/nltk_data...
[nltk_data]   Package brown is already up-to-date!


Total tagged sentences: 57340
Training sentences: 45872
Validation sentences: 5734
Test sentences: 5734
Vocabulary size: 24755
Tag set size: 12
Tags: ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']
Context window size: 2
Train examples: 929015
Validation examples: 116350
Test examples: 115827
torch.Size([929015, 5])
torch.Size([929015])
Data preparation complete.


In [6]:
embedding_paths = {
    "SVD": "/kaggle/input/datasets/akshat9303/embeddings/svd.pt",
    "Word2Vec": "/kaggle/input/datasets/akshat9303/embeddings/skipgram.pt",
    "GloVe": "/kaggle/input/datasets/akshat9303/embeddings/glove.pt"
}

results = {}
for name, path in embedding_paths.items():
    if os.path.exists(path):
        print(f"Loading {name} embeddings from {path}")
        embedding_data = torch.load(path)
        embedding_matrix = embedding_data['embeddings']
        embedding_matrix = embedding_matrix.to(device)
        print(f"Embedding matrix shape: {embedding_matrix.shape}")

        num_tags = len(unique_tags)
        model = MLPPosTagger(embedding_matrix, num_tags, window_size, freeze_embeddings=True).to(device)

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

        best_val_f1 = 0.0
        best_state = None
        epochs = 10

        for epoch in range(1, epochs + 1):
            model.train()
            total_loss = 0
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
            
            avg_loss = total_loss / len(train_loader)

            model.eval()
            all_preds = []
            all_labels = []

            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    outputs = model(X_batch)
                    preds = torch.argmax(outputs, dim=1)
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(y_batch.cpu().numpy())
            
            val_accuracy = accuracy_score(all_labels, all_preds)
            val_f1 = f1_score(all_labels, all_preds, average='macro')

            print(f"Epoch {epoch}/{epochs} - Train Loss: {avg_loss:.4f} - Val Accuracy: {val_accuracy:.4f} - Val F1: {val_f1:.4f}")

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state = model.state_dict()

        model.load_state_dict(best_state)
        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                preds = torch.argmax(outputs, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_batch.cpu().numpy())

        accuracy = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro')

        cm = confusion_matrix(all_labels, all_preds)
        print("Confusion Matrix:\n", cm)

        results[name] = {
            "accuracy": accuracy,
            "f1": f1,
            "y_true": all_labels,
            "y_pred": all_preds,
            "confusion_matrix": cm
        }

        print(f"{name} Test Accuracy: {accuracy:.4f}, "f"Macro-F1: {f1:.4f}")
    else:
        print(f"Embedding file {path} not found. Skipping.")

Loading SVD embeddings from /kaggle/input/datasets/akshat9303/embeddings/svd.pt
Embedding matrix shape: torch.Size([24867, 300])
Epoch 1/10 - Train Loss: 0.1255 - Val Accuracy: 0.9747 - Val F1: 0.9368
Epoch 2/10 - Train Loss: 0.0476 - Val Accuracy: 0.9750 - Val F1: 0.9408
Epoch 3/10 - Train Loss: 0.0296 - Val Accuracy: 0.9748 - Val F1: 0.9409
Epoch 4/10 - Train Loss: 0.0203 - Val Accuracy: 0.9737 - Val F1: 0.9332
Epoch 5/10 - Train Loss: 0.0149 - Val Accuracy: 0.9728 - Val F1: 0.9321
Epoch 6/10 - Train Loss: 0.0118 - Val Accuracy: 0.9723 - Val F1: 0.9249
Epoch 7/10 - Train Loss: 0.0097 - Val Accuracy: 0.9734 - Val F1: 0.9309
Epoch 8/10 - Train Loss: 0.0084 - Val Accuracy: 0.9723 - Val F1: 0.9296
Epoch 9/10 - Train Loss: 0.0076 - Val Accuracy: 0.9731 - Val F1: 0.9329
Epoch 10/10 - Train Loss: 0.0068 - Val Accuracy: 0.9715 - Val F1: 0.9284
Confusion Matrix:
 [[14790     0     0     0     0     0     0     0     0     0     0     1]
 [    0  7411    16   152     0     4   466    21     1 

In [7]:
print("\n" + "="*60)
print("FINAL POS TAGGING RESULTS (TEST SET)")
print("="*60)

print(f"{'Embedding':<15} {'Accuracy':<12} {'Macro-F1':<12}")
print("-"*60)

for name, res in results.items():
    print(f"{name:<15} {res['accuracy']:<12.4f} {res['f1']:<12.4f}")
    print(f"Confusion Matrix for {name}:\n{res['confusion_matrix']}\n")


FINAL POS TAGGING RESULTS (TEST SET)
Embedding       Accuracy     Macro-F1    
------------------------------------------------------------
SVD             0.9723       0.9218      
Confusion Matrix for SVD:
[[14790     0     0     0     0     0     0     0     0     0     0     1]
 [    0  7411    16   152     0     4   466    21     1     1   112    10]
 [    1     4 14227    52     3    34     8     2    26    80     7     0]
 [    0   125    87  5252     8     8    78     4     1    17    32     0]
 [    0     0     6     5  3738     3     0     0     1     0     0     0]
 [    1     0    23    13     1 13508     3     1    44     0     1     1]
 [    0   283     4    53     0     3 26675    89     3    13   263    39]
 [    0     7     1     0     0     1    73  1443     0     0    11     1]
 [    0     0    23     1     0    26     3     0  4975     1     1     0]
 [    0     5   125    16     0     0    23     0     0  2766     5     2]
 [    1    92    14    18     0     1   4

In [8]:

best_model = max(results, key=lambda k: results[k]["f1"])
print(f"\nBest model: {best_model}")

cm = confusion_matrix(
    results[best_model]["y_true"],
    results[best_model]["y_pred"]
)

print("Confusion Matrix:")
print(cm)


Best model: Word2Vec
Confusion Matrix:
[[14788     0     1     0     0     0     0     0     0     0     0     2]
 [    0  7430     5   121     2     1   507    11     0     1   101    15]
 [    2     8 14212    68     3    27     4     2    19    89     8     2]
 [    0   144    90  5207    11    19    92     3     0    18    27     1]
 [    0     0     3     4  3745     1     0     0     0     0     0     0]
 [    0     1    30     7     0 13519     2     1    33     0     2     1]
 [    0   278     7    27     0     5 26710    71     3    12   274    38]
 [    0    11     0     2     0     0    82  1432     0     0    10     0]
 [    0     0    25     0     0    31     3     0  4970     1     0     0]
 [    0     7   110    19     0     0    26     1     0  2770     7     2]
 [    1   111    15    15     0     1   445     8     2     3 17798     5]
 [    1     4     2     2     1     2    36     1     0     1     3    46]]


In [9]:
print('Done for the day!!!')

Done for the day!!!


In [10]:
for i, tag in enumerate(unique_tags):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - (TP + FP + FN)
    print(f"Tag: {tag}")
    print(f"  TP: {TP}, FP: {FP}, FN: {FN}, TN: {TN}")

Tag: .
  TP: 14788, FP: 4, FN: 3, TN: 101032
Tag: ADJ
  TP: 7430, FP: 564, FN: 764, TN: 107069
Tag: ADP
  TP: 14212, FP: 288, FN: 232, TN: 101095
Tag: ADV
  TP: 5207, FP: 265, FN: 405, TN: 109950
Tag: CONJ
  TP: 3745, FP: 17, FN: 8, TN: 112057
Tag: DET
  TP: 13519, FP: 87, FN: 77, TN: 102144
Tag: NOUN
  TP: 26710, FP: 1197, FN: 715, TN: 87205
Tag: NUM
  TP: 1432, FP: 98, FN: 105, TN: 114192
Tag: PRON
  TP: 4970, FP: 57, FN: 60, TN: 110740
Tag: PRT
  TP: 2770, FP: 125, FN: 172, TN: 112760
Tag: VERB
  TP: 17798, FP: 432, FN: 606, TN: 96991
Tag: X
  TP: 46, FP: 66, FN: 53, TN: 115662
